# Download e extração dos Microdados do ENADE 2023
Este notebook organiza a camada Bronze do projeto:
1. define as pastas do projeto;
2. inicia o download oficial do INEP;
3. preserva o arquivo ZIP original;
4. extrai e valida os arquivos recebidos.
Fonte: https://download.inep.gov.br/microdados/microdados_enade_2023.zip

# 1. Organização do projeto
Os caminhos são definidos de forma relativa. Assim, o notebook funciona na raiz do projeto ou dentro da pasta notebooks,

# 2. Download do arquivo
O Python abre o endereço oficial no navegador padrão. Como o link inicia o download diretamente, o notebook aguarda a conclusão e move o ZIP da pasta Downloads para data/raw.
Se o ZIP já estiver em data/raw, o download não será repetido.

In [1]:
#1. Organização do Projeto

from curl_cffi import requests as curl_requests
from pathlib import Path
from zipfile import ZipFile
import shutil
import time
import webbrowser
import pandas as pd

##1.1. URL de download

URL_ENADE = (
    "https://download.inep.gov.br/microdados/"
    "microdados_enade_2023.zip"
)

URL_CENSO = (
    "https://download.inep.gov.br/microdados/"
    "microdados_censo_da_educacao_superior_2023.zip"
)


##1.3. Identifica3 a pasta principal do projeto

pasta_atual = Path.cwd()

pasta_projeto = (
    pasta_atual.parent
    if pasta_atual.name == "notebooks"
    else pasta_atual
)


##1.4. Definir a organização das pastas e arquivos

pasta_raw = pasta_projeto / "data" / "raw"

#1.5. Cria data/raw caso ainda não exista

pasta_raw.mkdir(parents=True, exist_ok=True)

#1.6. Criar arquivos:

arquivo_zip_enade = pasta_raw / "microdados_enade_2023.zip"
arquivo_zip_censo = (
    pasta_raw / "microdados_censo_da_educacao_superior_2023.zip"
)

pasta_extraida_enade = pasta_raw / "microdados_enade_2023"
pasta_extraida_censo = (
    pasta_raw / "microdados_censo_da_educacao_superior_2023"
)


print(f"Pasta do projeto: {pasta_projeto}")
print(f"Pasta dos dados brutos: {pasta_raw}")


Pasta do projeto: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR
Pasta dos dados brutos: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\raw


In [2]:
#2. Download e extração dos arquivos

def baixar_arquivo(url, caminho_destino):
    """
    Baixa o arquivo em blocos e salva primeiro como .part.
    """

    if caminho_destino.exists() and caminho_destino.stat().st_size > 0:
        print(f"Arquivo já disponível: {caminho_destino}")
        return

    caminho_destino.parent.mkdir(parents=True, exist_ok=True)

    caminho_temporario = caminho_destino.with_name(
        caminho_destino.name + ".part"
    )

    # Remove uma tentativa de download incompleta
    caminho_temporario.unlink(missing_ok=True)

    print(f"Baixando: {caminho_destino.name}")

    resposta = None

    try:
        resposta = curl_requests.get(
            url,
            impersonate="chrome",
            stream=True,
            timeout=600
        )

        resposta.raise_for_status()

        with caminho_temporario.open("wb") as destino:
            for bloco in resposta.iter_content(
                chunk_size=1024 * 1024
            ):
                if bloco:
                    destino.write(bloco)

        caminho_temporario.replace(caminho_destino)

        print(f"Download concluído: {caminho_destino}")

    except Exception:
        caminho_temporario.unlink(missing_ok=True)
        raise

    finally:
        if resposta is not None:
            resposta.close()

    print(f"Download concluído: {caminho_destino}")


def extrair_zip(caminho_zip, pasta_destino):
    """
    Extrai o ZIP apenas se a pasta ainda não possuir arquivos.
    """

    pasta_destino.mkdir(parents=True, exist_ok=True)

    if any(pasta_destino.iterdir()):
        print(f"Arquivo já extraído: {pasta_destino}")
        return

    print(f"Extraindo: {caminho_zip.name}")

    with ZipFile(caminho_zip, "r") as arquivo_zip:
        arquivo_zip.extractall(pasta_destino)

    print(f"Extração concluída: {pasta_destino}")


def localizar_arquivo(pasta, nome_arquivo):
    """
    Procura um arquivo em todas as subpastas.
    """

    nome_arquivo = nome_arquivo.casefold()

    for arquivo in pasta.rglob("*"):
        if arquivo.is_file() and arquivo.name.casefold() == nome_arquivo:
            return arquivo

    return None


# 2.1. Download do arquivo ENADE

baixar_arquivo(
    URL_ENADE,
    arquivo_zip_enade
)


# 2.2. Download do arquivo Censo INEP

baixar_arquivo(
    URL_CENSO,
    arquivo_zip_censo
)

Arquivo já disponível: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\raw\microdados_enade_2023.zip
Arquivo já disponível: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\raw\microdados_censo_da_educacao_superior_2023.zip


# 3 - Extração dos microdados
A pasta de destino pode existir mesmo estando vazia. Por isso, a extração somente será ignorada quando houver arquivos dentro dela.

In [3]:
# 2.3. Extração dos arquivos ZIP

extrair_zip(
    arquivo_zip_enade,
    pasta_extraida_enade
)

extrair_zip(
    arquivo_zip_censo,
    pasta_extraida_censo
)


# 2.4. Localização do arquivo de cursos

nome_arquivo_cursos = "MICRODADOS_CADASTRO_CURSOS_2023.CSV"

arquivo_cursos = localizar_arquivo(
    pasta_extraida_censo,
    nome_arquivo_cursos
)

if arquivo_cursos is None:
    raise FileNotFoundError(
        f"O arquivo {nome_arquivo_cursos} não foi encontrado."
    )

pasta_dados = arquivo_cursos.parent

print("\nProcesso concluído.")
print(f"ZIP ENADE: {arquivo_zip_enade}")
print(f"ZIP Censo: {arquivo_zip_censo}")
print(f"Pasta dados: {pasta_dados}")
print(f"Arquivo CSV: {arquivo_cursos}")

Arquivo já extraído: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\raw\microdados_enade_2023
Arquivo já extraído: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\raw\microdados_censo_da_educacao_superior_2023

Processo concluído.
ZIP ENADE: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\raw\microdados_enade_2023.zip
ZIP Censo: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\raw\microdados_censo_da_educacao_superior_2023.zip
Pasta dados: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\raw\microdados_censo_da_educacao_superior_2023\microdados_censo_da_educacao_superior_2023\dados
Arquivo CSV: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\raw\microdados_censo_da_educacao_superior_2023\microdados_censo_da_educacao_superior_2023\dados\MICRODADOS_CADASTRO_CURSOS_2023.CSV


# 4 - Validação dos arquivos extraídos
A validação confirma que a camada Bronze contém arquivos e apresenta o inventário relativo à pasta de extração.

In [4]:
for nome, pasta in {
    "ENADE": pasta_extraida_enade,
    "CENSO": pasta_extraida_censo
}.items():

    arquivos = [
        arquivo
        for arquivo in pasta.rglob("*")
        if arquivo.is_file()
    ]

    if not arquivos:
        raise RuntimeError(f"Nenhum arquivo encontrado em {nome}.")

    print(f"{nome}: {len(arquivos)} arquivos encontrados.")


# Verifica o CSV necessário

arquivo_cursos = next(
    pasta_extraida_censo.rglob(
        "MICRODADOS_CADASTRO_CURSOS_2023.CSV"
    ),
    None
)

if arquivo_cursos is None:
    raise FileNotFoundError("Arquivo de cursos não encontrado.")

print(f"CSV encontrado: {arquivo_cursos}")

ENADE: 36 arquivos encontrados.
CENSO: 16 arquivos encontrados.
CSV encontrado: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\raw\microdados_censo_da_educacao_superior_2023\microdados_censo_da_educacao_superior_2023\dados\MICRODADOS_CADASTRO_CURSOS_2023.CSV
